# masks

> Clean up and encode segmentation masks. Works for binary masks and for multi-category (integer-labelled) masks.

Hand-drawn or weakly-labelled masks are often ragged. Two jobs live here: (1) snap connected components to clean shapes, and (2) convert between the integer-label representation used on disk and the one-hot representation used by losses.

In [ ]:
#| default_exp masks

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import cv2
import numpy as np

# cv2.drawContours silently no-ops on 64-bit Linux when passed int64 arrays.
# np.intp == int64 on 64-bit systems; np.int32 is safe on Linux AND Windows.
# Use this constant everywhere we pass contour coordinate arrays to OpenCV.
_CV_CONTOUR_DTYPE = np.int32

## A synthetic mask

We draw a rotated rectangle and a noisy blob so we can see the cleanup work. On real data, load with `cv2.imread(path, cv2.IMREAD_GRAYSCALE)`.

In [ ]:
raw = np.zeros((80, 80), np.uint8)
box = cv2.boxPoints(((30, 30), (24, 10), 25.0)).astype(np.int32)
cv2.fillPoly(raw, [box], 255)
rng = np.random.default_rng(0)
raw[55:65, 55:65] = (rng.random((10, 10)) > 0.4).astype(np.uint8) * 255  # ragged blob
print('unique values:', np.unique(raw))

## Binarise

Annotators save masks as `0/1` or `0/255`. `binarize` collapses either into a clean `{0, 1}` uint8.

In [ ]:
# experiment
b = (raw > 127).astype(np.uint8)
print('after binarise, unique:', np.unique(b))

In [ ]:
#| export
def binarize(
    mask: np.ndarray,    # mask with any positive encoding (1 or 255)
    thresh: int = 127,   # threshold when values are 0/255; ignored for 0/1 masks
) -> np.ndarray:
    """Collapse a mask to a `{0, 1}` uint8 array.

    Handles both `{0, 1}` and `{0, 255}` encodings automatically.

    Example:
        >>> binarize(np.array([[0, 255], [255, 0]], np.uint8)).tolist()
        [[0, 1], [1, 0]]
        >>> binarize(np.array([[0, 1], [1, 0]], np.uint8)).tolist()
        [[0, 1], [1, 0]]
    """
    if mask.max() <= 1:
        return (mask > 0).astype(np.uint8)
    return (mask > thresh).astype(np.uint8)

In [ ]:
# test
assert set(np.unique(binarize(raw)).tolist()) <= {0, 1}
assert binarize(np.array([[0, 1], [1, 0]], np.uint8)).tolist() == [[0, 1], [1, 0]]

## Snap blobs to rotated rectangles

Replace every connected component with the minimum-area rotated rectangle that encloses it. This removes ragged pixels and is the right prior when the targets are physically rectangular (pins, pads, dies).

In [ ]:
#| export
def fit_rotated_rects_mask(
    mask: np.ndarray,    # binary {0,1} (or 0/255) mask, any shape
    min_pts: int = 5,    # contours with fewer points are kept as-is
    min_area: int = 4,   # drop specks smaller than this (noise)
) -> np.ndarray:
    """Replace each connected blob with its min-area rotated bounding rectangle.

    Example:
        >>> clean = fit_rotated_rects_mask(np.eye(8, dtype=np.uint8))
        >>> clean.dtype, int(clean.max())
        (dtype('uint8'), 1)
    """
    m = binarize(mask)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    out = np.zeros_like(m, dtype=np.uint8)
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            continue
        if len(cnt) < min_pts:
            cv2.drawContours(out, [cnt], -1, 1, thickness=cv2.FILLED)
            continue
        box = cv2.boxPoints(cv2.minAreaRect(cnt)).astype(_CV_CONTOUR_DTYPE)
        cv2.drawContours(out, [box], 0, 1, thickness=cv2.FILLED)
    return out

def fit_rotated_rects_multiclass(
    mask: np.ndarray,    # integer-labelled mask (0 = background, 1..K = classes)
    num_classes: int,    # number of foreground classes K
    min_pts: int = 5,    # see fit_rotated_rects_mask
    min_area: int = 4,   # see fit_rotated_rects_mask
) -> np.ndarray:
    """Apply `fit_rotated_rects_mask` independently to each foreground label.

    Later labels win where rectangles overlap.
    """
    out = np.zeros_like(mask, dtype=mask.dtype)
    for label in range(1, num_classes + 1):
        cleaned = fit_rotated_rects_mask((mask == label).astype(np.uint8), min_pts, min_area)
        out[cleaned > 0] = label
    return out

In [ ]:
# experiment + test: cleanup fills the rotated rectangle and removes ragged noise
clean = fit_rotated_rects_mask(raw)
print('raw foreground px :', int(binarize(raw).sum()))
print('clean foreground px:', int(clean.sum()))
assert clean.dtype == np.uint8 and int(clean.max()) <= 1
# multiclass: two labels stay separate
mc = np.zeros((40, 40), np.uint8); mc[5:15, 5:15] = 1; mc[25:35, 25:35] = 2
mcc = fit_rotated_rects_multiclass(mc, num_classes=2)
assert set(np.unique(mcc).tolist()) == {0, 1, 2}

## One-hot encoding

Losses operate on per-class channels. `one_hot` turns an `(H, W)` integer-label mask into a `(num_classes, H, W)` float tensor-friendly array. For binary tasks (`num_classes == 1`) it returns the single foreground channel.

In [ ]:
#| export
def one_hot(
    mask: np.ndarray,   # (H, W) integer labels: 0 = background, 1..K = classes
    num_classes: int,   # K, the number of foreground classes
) -> np.ndarray:
    """Encode an integer-label mask as `(num_classes, H, W)` float32.

    Binary (`num_classes == 1`) returns the foreground channel only.
    Multi-class returns one channel per foreground label (background dropped).

    Example:
        >>> m = np.array([[0, 1], [2, 0]]); one_hot(m, 2).shape
        (2, 2, 2)
    """
    if num_classes == 1:
        return binarize(mask)[None].astype(np.float32)
    return np.stack([(mask == c) for c in range(1, num_classes + 1)], axis=0).astype(np.float32)

In [ ]:
# test
m = np.array([[0, 1], [2, 0]])
oh = one_hot(m, 2)
assert oh.shape == (2, 2, 2)
assert np.array_equal(oh.argmax(0) + 1, np.where(m > 0, m, 1))  # foreground labels recovered
assert one_hot(binarize(raw), 1).shape == (1, 80, 80)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()